# Session 2 — Infer with the fine-tuned Spring RAFT checkpoint

This notebook does not train. Attach the private Kaggle Dataset produced by Session 1; it must contain `raft-spring-left-finetuned.ckpt`. The notebook discovers that file, runs native-resolution inference on both cameras and temporal directions, validates every `.flo5`, and creates the official Spring benchmark HDF5 artifact.

Recommended accelerator: **GPU T4 x2**. The separately attached left/right test-frame datasets and `flow_subsampling` executable are also required.


In [ ]:
from pathlib import Path
import hashlib, importlib, json, os, platform, re, shutil, subprocess, sys, time

RUN_TRAINING = False
RUN_INFERENCE = True
RUN_PACKAGING = True
MODEL = "raft"
INFERENCE_ITERS = 32
INFERENCE_CORR_MODE = "triton"
INFERENCE_GPUS = 2
MAX_FORWARD_SIDE = None
DEVKIT_REF = "90ae81a9324c6806dc3c2482aab84a2744215bd9"
# Keep these aligned with the checkpoint-producing training notebook.
TRAINING_EPOCHS_METADATA = 1
TRAINING_ITERS_METADATA = 12

KAGGLE_INPUT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
SCRATCH_ROOT = Path("/kaggle/temp")
DEVKIT_DIR = WORK_ROOT / "roco-spring-devkit"
TEST_LEFT_ROOT = KAGGLE_INPUT / "datasets/strikingratio/test-frame-left-right/test_frame_left/spring"
TEST_RIGHT_ROOT = KAGGLE_INPUT / "datasets/strikingratio/test-frame-left-right/test_frame_right/spring"
PRETRAINED_CHECKPOINT = KAGGLE_INPUT / "datasets/strikingratio/ckpoint/raft-sintel-fb44381e.ckpt"
SUBSAMPLING_INPUT = KAGGLE_INPUT / "datasets/strikingratio/flow-subsampling/flow_subsampling"
EXPECTED_TRAIN_SEQUENCES = {"0011", "0022", "0025", "0026", "0027", "0030", "0032", "0036", "0041", "0045"}

# Usually leave this as None for automatic discovery by filename.
FINETUNED_CHECKPOINT_OVERRIDE = None

SPRING_TEST_ROOT = SCRATCH_ROOT / "spring_test_merged"
OUTPUT_BASE = SCRATCH_ROOT / "raft_spring_finetuned_predictions"
ARTIFACT_DIR = WORK_ROOT / "raft_spring_finetuned_artifacts"
SESSION_START = time.monotonic()
MAX_SESSION_HOURS = 12.0
PACKAGING_RESERVE_MINUTES = 30

assert KAGGLE_INPUT.is_dir(), "Run this notebook in Kaggle."
for directory in (WORK_ROOT, SCRATCH_ROOT):
    directory.mkdir(parents=True, exist_ok=True)


## 1. Locate the checkpoint exported by Session 1

If automatic discovery is ambiguous, set `FINETUNED_CHECKPOINT_OVERRIDE` in the configuration cell to its full attached-input path.


In [ ]:
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True, capture_output=True, check=True,
)
print(gpu.stdout)
gpu_count = len([line for line in gpu.stdout.splitlines() if line.strip()])
if gpu_count < 1:
    raise RuntimeError("Enable a Kaggle GPU accelerator and restart the session.")

# You may set FINETUNED_CHECKPOINT_OVERRIDE in the configuration cell. Otherwise,
# search only shallow input metadata and avoid descending into image folders.
if FINETUNED_CHECKPOINT_OVERRIDE:
    INFERENCE_CHECKPOINT = Path(FINETUNED_CHECKPOINT_OVERRIDE)
else:
    checkpoint_matches = []
    for current, dirs, files in os.walk(KAGGLE_INPUT):
        current = Path(current)
        try:
            depth = len(current.relative_to(KAGGLE_INPUT).parts)
        except ValueError:
            continue
        dirs[:] = [d for d in dirs if d not in {"frame_left", "frame_right", "flow_FW_left"}]
        if "raft-spring-left-finetuned.ckpt" in files:
            checkpoint_matches.append(current / "raft-spring-left-finetuned.ckpt")
        if depth >= 7:
            dirs[:] = []
    if len(checkpoint_matches) != 1:
        raise RuntimeError(
            "Attach exactly one training-output dataset containing "
            f"raft-spring-left-finetuned.ckpt; found {checkpoint_matches}"
        )
    INFERENCE_CHECKPOINT = checkpoint_matches[0]

if not INFERENCE_CHECKPOINT.is_file() or INFERENCE_CHECKPOINT.stat().st_size < 1_000_000:
    raise RuntimeError(f"Fine-tuned checkpoint is missing or incomplete: {INFERENCE_CHECKPOINT}")
print("Fine-tuned checkpoint:", INFERENCE_CHECKPOINT)
print("Python:", sys.version.split()[0], "OS:", platform.platform())


## 2. Install the pinned official devkit

The same pinned revision as the original baseline is used. With Kaggle Internet disabled, attach a copy of the devkit containing `roco_spring_devkit/optical_flow/train.py`; otherwise the cell clones it.


In [ ]:
def shallow_walk(root: Path, max_depth=5):
    root = root.resolve()
    for current, dirs, files in os.walk(root):
        current = Path(current)
        yield current, dirs, files
        if len(current.relative_to(root).parts) >= max_depth:
            dirs[:] = []

def find_attached_devkit():
    for current, _, files in shallow_walk(KAGGLE_INPUT):
        if "pyproject.toml" in files and (current / "roco_spring_devkit/optical_flow/train.py").is_file():
            return current
    return None

if not (DEVKIT_DIR / "roco_spring_devkit/optical_flow/train.py").is_file():
    attached = find_attached_devkit()
    if attached:
        print("Copying attached devkit:", attached)
        shutil.copytree(attached, DEVKIT_DIR, dirs_exist_ok=True)
    else:
        print("Cloning the official devkit; Kaggle Internet must be enabled.")
        subprocess.run(["git", "clone", "https://github.com/hmorimitsu/roco-spring-devkit.git", str(DEVKIT_DIR)], check=True)
if (DEVKIT_DIR / ".git").is_dir():
    subprocess.run(["git", "checkout", DEVKIT_REF], cwd=DEVKIT_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(DEVKIT_DIR)], check=True)
devkit_path = str(DEVKIT_DIR.resolve())
if devkit_path not in sys.path:
    sys.path.insert(0, devkit_path)
importlib.invalidate_caches()

from roco_spring_devkit.optical_flow.models.raft.raft import RAFT
probe = RAFT(iters=INFERENCE_ITERS, corr_mode=INFERENCE_CORR_MODE, predict_all_directions=True)
parameter_count = sum(p.numel() for p in probe.parameters())
print(f"Devkit ready; full RAFT has {parameter_count / 1e6:.2f}M parameters")
del probe


## 4. Assemble and verify the stereo test tree

The test images are attached as separate left/right roots. Directory symlinks build the unified layout expected by the official loader without copying thousands of PNGs.


In [ ]:
EXPECTED_SEQUENCES = {"0003", "0019", "0028", "0029", "0031", "0034", "0035", "0040", "0042", "0046"}

def ensure_link(source: Path, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_symlink():
        if destination.resolve() != source.resolve():
            raise RuntimeError(f"Conflicting symlink: {destination}")
    elif destination.exists():
        raise RuntimeError(f"Destination already exists and is not a symlink: {destination}")
    else:
        destination.symlink_to(source.resolve(), target_is_directory=True)

expected_flow_files = 0
if RUN_INFERENCE:
    for root in (TEST_LEFT_ROOT, TEST_RIGHT_ROOT):
        if not root.is_dir():
            raise FileNotFoundError(root)
    for sequence_name in sorted(EXPECTED_SEQUENCES):
        left = TEST_LEFT_ROOT / "test" / sequence_name / "frame_left"
        right = TEST_RIGHT_ROOT / "test" / sequence_name / "frame_right"
        left_frames = sorted(left.glob("frame_left_*.png"))
        right_frames = sorted(right.glob("frame_right_*.png"))
        if not left_frames or len(left_frames) != len(right_frames):
            raise RuntimeError(f"Missing/mismatched test frames for {sequence_name}: {len(left_frames)} vs {len(right_frames)}")
        ensure_link(left, SPRING_TEST_ROOT / "test" / sequence_name / "frame_left")
        ensure_link(right, SPRING_TEST_ROOT / "test" / sequence_name / "frame_right")
        expected_flow_files += 4 * (len(left_frames) - 1)
    print(f"Verified {len(EXPECTED_SEQUENCES)} test sequences; expecting {expected_flow_files:,} prediction files")
else:
    print("Inference skipped; test tree was not assembled.")


## 5. Run fine-tuned native-resolution inference

This uses 32 RAFT updates, both temporal directions and both cameras. No scale argument is passed, so saved flow remains `1080×1920×2`. The timeout preserves time for validation and packaging.


In [ ]:
if RUN_INFERENCE:
    inference_gpus = min(INFERENCE_GPUS, gpu_count)
    infer_env = os.environ.copy()
    infer_env["CUDA_VISIBLE_DEVICES"] = ",".join(str(i) for i in range(inference_gpus))
    infer_env["PYTHONUNBUFFERED"] = "1"
    infer_env["PYTHONPATH"] = devkit_path + (os.pathsep + infer_env["PYTHONPATH"] if infer_env.get("PYTHONPATH") else "")
    OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, "test.py",
        "--data.test_dataset", "spring",
        "--data.spring_root_dir", str(SPRING_TEST_ROOT),
        "--model", MODEL,
        "--ckpt_path", str(INFERENCE_CHECKPOINT),
        "--model.corr_mode", INFERENCE_CORR_MODE,
        "--model.iters", str(INFERENCE_ITERS),
        "--model.predict_all_directions", "true",
        "--num_gpus", str(inference_gpus),
        "--output_path", str(OUTPUT_BASE),
    ]
    assert MAX_FORWARD_SIDE is None
    assert "--max_forward_side" not in command and "--scale_factor" not in command
    print("Command:", " ".join(command))
    remaining = MAX_SESSION_HOURS * 3600 - PACKAGING_RESERVE_MINUTES * 60 - (time.monotonic() - SESSION_START)
    if remaining <= 0:
        raise TimeoutError("No safe inference time remains; use the two-session workflow described above.")
    subprocess.run(
        command,
        cwd=DEVKIT_DIR / "roco_spring_devkit/optical_flow",
        env=infer_env,
        check=True,
        timeout=remaining,
    )
else:
    print("Inference skipped. Save", FINETUNED_CHECKPOINT, "as a Kaggle Dataset for a later session.")


## 6. Validate and package predictions

Every expected `.flo5` is checked for naming and native shape. The official `flow_subsampling` executable then creates the benchmark HDF5 artifact. A manifest records both the initialization and fine-tuned checkpoint hashes.


In [ ]:
if RUN_INFERENCE:
    import h5py
    import numpy as np

    prediction_roots = [p for p in OUTPUT_BASE.glob("*/spring") if any(p.rglob("*.flo5"))]
    if len(prediction_roots) != 1:
        raise RuntimeError(f"Expected one Spring prediction tree; found {prediction_roots}")
    PREDICTION_ROOT = prediction_roots[0]
    prediction_files = sorted(PREDICTION_ROOT.rglob("*.flo5"))
    if len(prediction_files) != expected_flow_files:
        raise RuntimeError(f"Expected {expected_flow_files} flow files; found {len(prediction_files)}")

    name_re = re.compile(r"flow_(FW|BW)_(left|right)_\d{4}\.flo5$")
    combinations = set()
    for path in prediction_files:
        match = name_re.fullmatch(path.name)
        if not match:
            raise RuntimeError(f"Invalid prediction filename: {path}")
        combinations.add((match.group(1), match.group(2)))
        with h5py.File(path, "r") as handle:
            shape = handle["flow"].shape if "flow" in handle else None
            if shape != (1080, 1920, 2):
                raise RuntimeError(f"Invalid native flow shape in {path}: {shape}")
    required_combinations = {(d, s) for d in ("FW", "BW") for s in ("left", "right")}
    if combinations != required_combinations:
        raise RuntimeError(f"Missing direction/view outputs: {sorted(combinations)}")
    for index in sorted({0, len(prediction_files) // 2, len(prediction_files) - 1}):
        with h5py.File(prediction_files[index], "r") as handle:
            if not np.isfinite(handle["flow"][:]).all():
                raise RuntimeError(f"NaN/Inf detected in {prediction_files[index]}")
    print(f"Validated {len(prediction_files):,} native-resolution predictions")
else:
    prediction_files = []


In [ ]:
def sha256(path: Path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if RUN_INFERENCE and RUN_PACKAGING:
    if not SUBSAMPLING_INPUT.is_file():
        raise FileNotFoundError(SUBSAMPLING_INPUT)
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    tool = WORK_ROOT / "flow_subsampling"
    shutil.copy2(SUBSAMPLING_INPUT, tool)
    tool.chmod(0o755)
    subprocess.run([str(tool), str(PREDICTION_ROOT)], cwd=ARTIFACT_DIR, check=True)
    submissions = sorted(ARTIFACT_DIR.glob("*.hdf5"), key=lambda p: p.stat().st_mtime)
    if not submissions:
        raise RuntimeError("The official packager did not create an HDF5 file.")
    submission = submissions[-1]
    run_manifest = {
        "model": "full RAFT",
        "initial_checkpoint": str(PRETRAINED_CHECKPOINT),
        "initial_checkpoint_sha256": sha256(PRETRAINED_CHECKPOINT) if PRETRAINED_CHECKPOINT.is_file() else None,
        "fine_tuned_checkpoint": str(INFERENCE_CHECKPOINT),
        "fine_tuned_checkpoint_sha256": sha256(INFERENCE_CHECKPOINT),
        "train_sequences": sorted(EXPECTED_TRAIN_SEQUENCES - {"0022"}),
        "validation_sequences": ["0022"],
        "training_epochs": TRAINING_EPOCHS_METADATA,
        "training_iterations": TRAINING_ITERS_METADATA,
        "inference_iterations": INFERENCE_ITERS,
        "inference_correlation": INFERENCE_CORR_MODE,
        "resolution": "1920x1080_native",
        "devkit_ref": DEVKIT_REF,
        "prediction_files": len(prediction_files),
        "submission_file": submission.name,
        "submission_sha256": sha256(submission),
    }
    manifest_path = ARTIFACT_DIR / "raft_spring_finetuned_manifest.json"
    manifest_path.write_text(json.dumps(run_manifest, indent=2) + "\n")
    print(manifest_path.read_text())
    print("UPLOAD THIS FILE:", submission)
    print("KEEP THIS CHECKPOINT:", FINETUNED_CHECKPOINT)
elif RUN_TRAINING:
    print("Training output checkpoint:", FINETUNED_CHECKPOINT)


## Output

Download the generated `.hdf5` and `raft_spring_finetuned_manifest.json` from `/kaggle/working/raft_spring_finetuned_artifacts`. Upload the HDF5 file to the Spring optical-flow benchmark.
